In [ ]:
# ── Imports and configuration ──

import pandas as pd
import numpy as np
import pickle
import os
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, classification_report

import time
import shap

# ── Paths ───────────────────────────────────────────────────────────────────
BASE_DIR = Path(".")
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Input files ─────────────────────────────────────────────────────────────
CLR_PATH        = OUTPUT_DIR / "preprocessed_otu_clr.csv"
DELTA_PATH      = OUTPUT_DIR / "delta_otu_clr.csv"
METADATA_PATH   = OUTPUT_DIR / "merged_metadata_clean.csv"

# ── Output files ─────────────────────────────────────────────────────────────
RF_MODEL_PATH   = OUTPUT_DIR / "rf_model.pkl"
FI_PATH         = OUTPUT_DIR / "feature_importances.csv"
SHAP_PATH       = OUTPUT_DIR / "shap_values.csv"
REPORT_PATH     = OUTPUT_DIR / "classification_report.txt"

# ── Random seed ──────────────────────────────────────────────────────────────
SEED = 42

# ── Hyperparameter grid ──────────────────────────────────────────────────────
PARAM_GRID = {
    "n_estimators":     [100, 300, 500],
    "max_features":     ["sqrt", "log2", 0.1],
    "min_samples_leaf": [1, 3],
}

# ── SHAP subset size ─────────────────────────────────────────────────────────
SHAP_N = 250   # samples to run SHAP on — increase if runtime allows

print("Config OK")
print(f"Output dir: {OUTPUT_DIR}")

In [ ]:
# ── Load data and build model datasets ──

# ── Load files ───────────────────────────────────────────────────────────────
clr      = pd.read_csv(CLR_PATH,      index_col=0)
delta    = pd.read_csv(DELTA_PATH,    index_col=[0, 1, 2])
metadata = pd.read_csv(METADATA_PATH, index_col=0)

print(f"CLR matrix:   {clr.shape}")
print(f"Delta matrix: {delta.shape}")
print(f"Metadata:     {metadata.shape}")

# ── Model A: fiber-arm only, timepoint before=0 / after=1 ───────────────────
meta_fiber = metadata[metadata["treatment"] == "fiber"].copy()
clr_fiber  = clr.loc[clr.index.intersection(meta_fiber.index)]
meta_fiber = meta_fiber.loc[clr_fiber.index]

X_A      = clr_fiber
y_A      = (meta_fiber["timepoint"] == "after").astype(int)
groups_A = meta_fiber["study"]
subjects_A = meta_fiber["subject_id"]

print(f"\nModel A — fiber arm, before vs after:")
print(f"  Samples:  {X_A.shape[0]}  |  Features: {X_A.shape[1]}")
print(f"  Before:   {(y_A==0).sum()}  |  After: {y_A.sum()}")
print(f"  Studies:  {groups_A.nunique()}")
print(f"  Subjects: {subjects_A.nunique()}")

# ── Control-arm validation set (applied post-hoc, not used in training) ──────
meta_ctrl = metadata[metadata["treatment"] == "control"].copy()
clr_ctrl  = clr.loc[clr.index.intersection(meta_ctrl.index)]
meta_ctrl = meta_ctrl.loc[clr_ctrl.index]

X_ctrl      = clr_ctrl
y_ctrl      = (meta_ctrl["timepoint"] == "after").astype(int)
groups_ctrl = meta_ctrl["study"]

print(f"\nControl-arm validation (post-hoc only):")
print(f"  Samples:  {X_ctrl.shape[0]}  |  Features: {X_ctrl.shape[1]}")
print(f"  Before:   {(y_ctrl==0).sum()}  |  After: {y_ctrl.sum()}")
print(f"  Studies:  {groups_ctrl.nunique()}")

In [ ]:
# ── LOSO-CV with inner grid search ──
def run_loso_cv(X, y, groups, subjects, label="Model"):
    """
    Leave-One-Study-Out CV with subject-level awareness.
    Inner grid search on 5-fold stratified CV within training set.
    """
    studies = groups.unique()
    results = []
    all_best_params = []

    print(f"\n{'='*60}")
    print(f"{label} — LOSO-CV ({len(studies)} folds)")
    print(f"{'='*60}")

    for study in studies:
        t0 = time.time()

        # ── Outer split ──────────────────────────────────────────────────────
        test_mask  = (groups == study).values
        train_mask = ~test_mask

        X_train, X_test = X.values[train_mask], X.values[test_mask]
        y_train, y_test = y.values[train_mask], y.values[test_mask]

        if len(np.unique(y_test)) < 2:
            print(f"  [{study}] SKIP — only one class in test fold")
            continue

        # ── Inner grid search ────────────────────────────────────────────────
        inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

        base_rf = RandomForestClassifier(
            class_weight="balanced",
            random_state=SEED,
            n_jobs=-1
        )

        grid = GridSearchCV(
            estimator=base_rf,
            param_grid=PARAM_GRID,
            cv=inner_cv,
            scoring="roc_auc",
            n_jobs=-1,
            refit=True,
            verbose=0
        )
        grid.fit(X_train, y_train)

        best_rf     = grid.best_estimator_
        best_params = grid.best_params_
        all_best_params.append(best_params)

        # ── Evaluate on left-out study ───────────────────────────────────────
        y_prob = best_rf.predict_proba(X_test)[:, 1]
        y_pred = best_rf.predict(X_test)

        auc = roc_auc_score(y_test, y_prob)
        f1  = f1_score(y_test, y_pred, average="weighted", zero_division=0)
        acc = accuracy_score(y_test, y_pred)

        elapsed = time.time() - t0

        results.append({
            "study":                 study,
            "n_test":                int(test_mask.sum()),
            "n_before":              int((y_test == 0).sum()),
            "n_after":               int((y_test == 1).sum()),
            "AUC":                   round(auc, 4),
            "F1_weighted":           round(f1, 4),
            "Accuracy":              round(acc, 4),
            "best_n_estimators":     best_params["n_estimators"],
            "best_max_features":     best_params["max_features"],
            "best_min_samples_leaf": best_params["min_samples_leaf"],
            "time_sec":              round(elapsed, 1),
        })

        print(f"  [{study:20s}]  AUC={auc:.3f}  F1={f1:.3f}  Acc={acc:.3f}"
              f"  n_test={test_mask.sum():4d}  best={best_params}  ({elapsed:.0f}s)")

    results_df = pd.DataFrame(results)

    print(f"\n── Summary ─────────────────────────────────────────────")
    print(f"  Folds evaluated : {len(results_df)}")
    print(f"  Mean AUC        : {results_df['AUC'].mean():.4f} ± {results_df['AUC'].std():.4f}")
    print(f"  Mean F1         : {results_df['F1_weighted'].mean():.4f} ± {results_df['F1_weighted'].std():.4f}")
    print(f"  Mean Accuracy   : {results_df['Accuracy'].mean():.4f} ± {results_df['Accuracy'].std():.4f}")

    # ── Most common best params across folds ─────────────────────────────────
    params_df = pd.DataFrame(all_best_params)
    print(f"\n── Most frequent best hyperparameters across folds ─────")
    for col in params_df.columns:
        most_common = params_df[col].value_counts().idxmax()
        print(f"  {col:25s}: {most_common}")

    return results_df, params_df


# ── Run Model A ──────────────────────────────────────────────────────────────
results_A, params_A = run_loso_cv(
    X_A, y_A, groups_A, subjects_A,
    label="Model A (fiber arm, before vs after)"
)

In [ ]:
# ── Final model fit and control-arm validation ──

# ── Fit final model on all fiber-arm data ────────────────────────────────────
print("Fitting final model on all fiber-arm data...")
print(f"  Training samples : {X_A.shape[0]}")
print(f"  Features         : {X_A.shape[1]}")
print(f"  Params           : n_estimators=500, max_features=0.1, min_samples_leaf=1")
t0 = time.time()

final_rf = RandomForestClassifier(
    n_estimators=500,
    max_features=0.1,
    min_samples_leaf=1,
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1
)
final_rf.fit(X_A.values, y_A.values)
print(f"  Done. ({time.time()-t0:.0f}s)\n")

# ── Save model ───────────────────────────────────────────────────────────────
with open(RF_MODEL_PATH, "wb") as f:
    pickle.dump(final_rf, f)
print(f"Model saved -> {RF_MODEL_PATH}\n")

# ── Control-arm validation ───────────────────────────────────────────────────
print("-- Control-arm validation ------------------------------------------")
print("  Applying fiber-trained model to control-arm samples (not seen during training)")
print(f"  Control samples : {X_ctrl.shape[0]}  |  Studies : {groups_ctrl.nunique()}\n")

ctrl_results = []
for study in groups_ctrl.unique():
    mask = (groups_ctrl == study).values
    X_c  = X_ctrl.values[mask]
    y_c  = y_ctrl.values[mask]

    if len(np.unique(y_c)) < 2:
        print(f"  [{study}] SKIP -- only one class")
        continue

    print(f"  [{study:20s}] predicting...", end=" ")
    y_prob_c = final_rf.predict_proba(X_c)[:, 1]
    y_pred_c = final_rf.predict(X_c)

    auc_c = roc_auc_score(y_c, y_prob_c)
    f1_c  = f1_score(y_c, y_pred_c, average="weighted", zero_division=0)
    acc_c = accuracy_score(y_c, y_pred_c)

    ctrl_results.append({
        "study":       study,
        "n_test":      int(mask.sum()),
        "n_before":    int((y_c == 0).sum()),
        "n_after":     int((y_c == 1).sum()),
        "AUC":         round(auc_c, 4),
        "F1_weighted": round(f1_c, 4),
        "Accuracy":    round(acc_c, 4),
    })

    print(f"AUC={auc_c:.3f}  F1={f1_c:.3f}  Acc={acc_c:.3f}  n={mask.sum()}")

ctrl_df = pd.DataFrame(ctrl_results)

print(f"\n-- Control-arm summary ---------------------------------------------")
print(f"  Mean AUC : {ctrl_df['AUC'].mean():.4f} +/- {ctrl_df['AUC'].std():.4f}")
print(f"  Mean F1  : {ctrl_df['F1_weighted'].mean():.4f} +/- {ctrl_df['F1_weighted'].std():.4f}")
print(f"  Fiber LOSO mean AUC : {results_A['AUC'].mean():.4f}")
print(f"  Control mean AUC    : {ctrl_df['AUC'].mean():.4f}")

# ── Save classification report ───────────────────────────────────────────────
print("\nSaving classification report...", end=" ")
with open(REPORT_PATH, "w", encoding="utf-8") as f:
    f.write("=" * 60 + "\n")
    f.write("RANDOM FOREST CLASSIFICATION REPORT\n")
    f.write("Temporal drift confound correction — RF classification report\n")
    f.write("=" * 60 + "\n\n")

    f.write("MODEL: Fiber-arm, before (0) vs after (1)\n")
    f.write("FEATURES: Batch-corrected CLR-transformed OTU abundances\n")
    f.write("FINAL MODEL PARAMS: n_estimators=500, max_features=0.1, "
            "min_samples_leaf=1, class_weight=balanced\n\n")

    f.write("-- LOSO-CV Results (fiber arm) --\n")
    f.write(results_A.to_string(index=False))
    f.write(f"\n\nMean AUC : {results_A['AUC'].mean():.4f} +/- {results_A['AUC'].std():.4f}")
    f.write(f"\nMean F1  : {results_A['F1_weighted'].mean():.4f} +/- {results_A['F1_weighted'].std():.4f}")
    f.write(f"\nMean Acc : {results_A['Accuracy'].mean():.4f} +/- {results_A['Accuracy'].std():.4f}\n")

    f.write("\n-- Control-arm Validation --\n")
    f.write(ctrl_df.to_string(index=False))
    f.write(f"\n\nMean AUC : {ctrl_df['AUC'].mean():.4f} +/- {ctrl_df['AUC'].std():.4f}")
    f.write(f"\nMean F1  : {ctrl_df['F1_weighted'].mean():.4f} +/- {ctrl_df['F1_weighted'].std():.4f}\n")

print(f"Done -> {REPORT_PATH}")

In [ ]:
# ── MDI feature importance ──

print("Extracting MDI feature importances...")

# ── Extract MDI from final model ─────────────────────────────────────────────
mdi = final_rf.feature_importances_
feature_names = X_A.columns.tolist()

fi_df = pd.DataFrame({
    "OTU_ID":     feature_names,
    "MDI":        mdi,
}).sort_values("MDI", ascending=False).reset_index(drop=True)

fi_df["MDI_rank"] = fi_df.index + 1

print(f"  Total features : {len(fi_df)}")
print(f"  Top 10 OTUs by MDI:")
print(fi_df[["MDI_rank", "OTU_ID", "MDI"]].head(10).to_string(index=False))

# ── Load taxonomy table for annotation ───────────────────────────────────────
print("\nLoading taxonomy table...", end=" ")
tax_path = OUTPUT_DIR / "taxonomy_table.csv"
tax_df   = pd.read_csv(tax_path, index_col=0)
tax_df.index.name = "OTU_ID"
tax_df = tax_df.reset_index()
print(f"Done. ({len(tax_df)} OTUs)")

# ── Merge taxonomy into feature importance ───────────────────────────────────
fi_df = fi_df.merge(tax_df, on="OTU_ID", how="left")

# ── Top 50 ───────────────────────────────────────────────────────────────────
top50 = fi_df.head(50).copy()

print(f"\nTop 50 OTUs — MDI importance + taxonomy:")
# Print whichever taxonomy column exists
tax_col = [c for c in tax_df.columns if c != "OTU_ID"][0]
print(top50[["MDI_rank", "OTU_ID", "MDI", tax_col]].to_string(index=False))

# ── Save ─────────────────────────────────────────────────────────────────────
print(f"\nSaving feature importances...", end=" ")
fi_df.to_csv(FI_PATH, index=False)
print(f"Done -> {FI_PATH}")
print(f"  Full table : {len(fi_df)} OTUs")
print(f"  Top 50 saved within same file (MDI_rank 1-50)")

In [ ]:
# ── SHAP analysis ──

print("Running SHAP TreeExplainer on final model...")
print(f"  Using stratified subset of {SHAP_N} samples from fiber-arm data\n")

# ── Stratified subset ────────────────────────────────────────────────────────
# Sample equal numbers of before/after to avoid class bias in SHAP values
np.random.seed(SEED)

before_idx = np.where(y_A.values == 0)[0]
after_idx  = np.where(y_A.values == 1)[0]

n_each = SHAP_N // 2
before_sample = np.random.choice(before_idx, size=min(n_each, len(before_idx)), replace=False)
after_sample  = np.random.choice(after_idx,  size=min(n_each, len(after_idx)),  replace=False)

subset_idx = np.concatenate([before_sample, after_sample])
X_shap = X_A.values[subset_idx]
y_shap = y_A.values[subset_idx]

print(f"  Subset: {len(subset_idx)} samples  "
      f"(before={len(before_sample)}, after={len(after_sample)})")

# ── SHAP TreeExplainer ───────────────────────────────────────────────────────
print("  Initializing TreeExplainer...", end=" ")
explainer = shap.TreeExplainer(final_rf)
print("Done.")

print("  Computing SHAP values (this may take 10-20 min on CPU)...")
t0 = time.time()
shap_values = explainer.shap_values(X_shap, check_additivity=False)
print(f"  Done. ({time.time()-t0:.0f}s)")

# ── shap_values is a list of 2 arrays [class_0, class_1] ─────────────────────
# Use class_1 (after=1) SHAP values — positive = pushes toward "after"
print("\n  SHAP values shape:", np.array(shap_values).shape)
shap_after = shap_values[1]   # shape: (n_samples, n_features)

# ── Mean absolute SHAP per feature ───────────────────────────────────────────
mean_abs_shap = np.abs(shap_after).mean(axis=0)

shap_df = pd.DataFrame({
    "OTU_ID":        X_A.columns.tolist(),
    "mean_abs_SHAP": mean_abs_shap,
}).sort_values("mean_abs_SHAP", ascending=False).reset_index(drop=True)

shap_df["SHAP_rank"] = shap_df.index + 1

# ── Merge taxonomy ───────────────────────────────────────────────────────────
shap_df = shap_df.merge(tax_df, on="OTU_ID", how="left")

# ── Cross-reference with MDI ranks ───────────────────────────────────────────
shap_df = shap_df.merge(
    fi_df[["OTU_ID", "MDI_rank", "MDI"]],
    on="OTU_ID", how="left"
)

top50_shap = shap_df.head(50).copy()

print(f"\nTop 20 OTUs by mean |SHAP|:")
tax_col = [c for c in tax_df.columns if c != "OTU_ID"][0]
print(top50_shap[["SHAP_rank", "MDI_rank", "OTU_ID",
                   "mean_abs_SHAP", tax_col]].head(20).to_string(index=False))

# ── Convergence check: how many top-20 SHAP OTUs are also top-20 MDI ─────────
shap_top20_otus = set(top50_shap.head(20)["OTU_ID"])
mdi_top20_otus  = set(fi_df.head(20)["OTU_ID"])
overlap = shap_top20_otus & mdi_top20_otus

print(f"\nConvergence — top-20 overlap between SHAP and MDI: "
      f"{len(overlap)}/20 OTUs")
print(f"  Overlapping OTUs: {overlap}")

# ── Save SHAP values for top 50 OTUs ─────────────────────────────────────────
print(f"\nSaving SHAP outputs...", end=" ")

# Full mean |SHAP| table
shap_df.to_csv(SHAP_PATH, index=False)

# SHAP matrix for top 50 OTUs (samples x OTUs) — for figures in Stage 7
top50_otu_ids  = top50_shap["OTU_ID"].tolist()
top50_otu_idx  = [X_A.columns.tolist().index(o) for o in top50_otu_ids]
shap_matrix_top50 = pd.DataFrame(
    shap_after[:, top50_otu_idx],
    columns=top50_otu_ids
)
shap_matrix_top50.insert(0, "timepoint", y_shap)
shap_matrix_top50.to_csv(OUTPUT_DIR / "shap_matrix_top50.csv", index=False)

print(f"Done.")
print(f"  Mean |SHAP| table -> {SHAP_PATH}")
print(f"  SHAP matrix (top 50) -> {OUTPUT_DIR / 'shap_matrix_top50.csv'}")